In [1]:
import pandas as pd
import os

In [2]:
!pip install --upgrade pandas pyarrow

In [2]:
# Create a dictionary to store each DataFrame, using the file name as the key

dataframes = {}
parquet_directory_path = "./sequential/50m" 

for file in os.listdir(parquet_directory_path):
    if file.endswith('.parquet'):
        file_path = os.path.join(parquet_directory_path, file)
        df = pd.read_parquet(file_path)
        dataframes[file] = df

# Now, dataframes['filename.parquet'] gives you the DataFrame for each file
print(dataframes.keys()) 

dict_keys(['undislikes.parquet', 'likes.parquet', 'unlikes.parquet', 'artist_item_mapping.parquet', 'multi_event.parquet', 'listens.parquet', 'dislikes.parquet'])


In [3]:
dataframes['listens.parquet'].head()

,uid,timestamp,item_id,is_organic,played_ratio_pct,track_length_seconds
0,100,"[39420, 39420, 39625, 40110, 40360, 40380, 406...","[8326270, 1441281, 286361, 732449, 3397170, 78...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[100, 100, 100, 100, 46, 100, 100, 100, 100, 1...","[170, 105, 185, 240, 130, 205, 205, 145, 95, 2..."
1,200,"[14329075, 14329075, 14329075, 14329075, 14329...","[3285270, 5253582, 9155883, 6113313, 1142869, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[9, 28, 42, 32, 0, 0, 5, 9, 100, 100, 100, 6, ...","[170, 170, 205, 155, 215, 185, 205, 195, 165, ..."
2,300,"[54090, 54100, 54110, 54120, 54125, 54135, 541...","[618910, 8793425, 8842650, 5806319, 8495320, 9...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[2, 4, 4, 5, 1, 5, 7, 1, 2, 2, 10, 6, 1, 23, 6...","[270, 130, 225, 210, 300, 210, 190, 340, 240, ..."
3,500,"[22695440, 22695690, 22695715, 22696095, 22696...","[6417502, 6896222, 6896938, 6950641, 7981221, ...","[0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, ...","[100, 37, 10, 37, 100, 100, 11, 13, 20, 100, 1...","[225, 210, 240, 165, 150, 245, 135, 165, 275, ..."
4,600,"[1329190, 1329405, 1329690, 1329950, 1330185, ...","[8077497, 1865247, 95237, 4624864, 8332575, 44...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[100, 100, 100, 100, 100, 100, 100, 100, 100, ...","[245, 215, 235, 260, 250, 280, 180, 195, 155, ..."


In [4]:
print(dataframes['listens.parquet'].shape)

(9238, 6)


In [5]:
dataframes['likes.parquet'].head()

,uid,timestamp,item_id,is_organic
0,100,"[44755, 1155860, 1259125, 1260005, 1263935, 20...","[732449, 6568592, 5411243, 7371186, 4943655, 4...","[1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, ..."
1,300,"[54135, 54285, 54325, 944000, 944170, 944215, ...","[9380688, 9104028, 7237185, 717747, 4408525, 6...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
2,400,"[3417540, 3691115, 5250445, 5832995, 5840165, ...","[1575108, 7443269, 3245239, 8383403, 1309246, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
3,500,"[23942600, 24861395, 24861430, 24861430]","[8195897, 1005397, 4077285, 3375935]","[1, 1, 1, 1]"
4,700,"[15450390, 25939545]","[4161409, 3383415]","[1, 0]"


In [6]:
print(len(dataframes['listens.parquet']['is_organic'][0]))
print(len(dataframes['likes.parquet']['uid'].unique()))

3585
8283


In [7]:
def clean_is_organic(df):
    # Remove entries where is_organic == 0 from all relevant columns
    def filter_row(row):
        mask = [flag == 1 for flag in row['is_organic']]
        row['is_organic'] = [flag for flag, keep in zip(row['is_organic'], mask) if keep]
        row['item_id'] = [item for item, keep in zip(row['item_id'], mask) if keep]
        row['timestamp'] = [ts for ts, keep in zip(row['timestamp'], mask) if keep]
        row['played_ratio_pct'] = [ratio for ratio, keep in zip(row['played_ratio_pct'], mask) if keep]

        return row

    return df.apply(filter_row, axis=1)

organic_listen = clean_is_organic(dataframes['listens.parquet'])
organic_listen.head()

,uid,timestamp,item_id,is_organic,played_ratio_pct,track_length_seconds
0,100,"[104725, 287910, 1265850, 1265990, 1266160, 12...","[732449, 5699545, 4943655, 8046835, 8661238, 4...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 100, 100, 100, 100, 100, 100, 100, 100, 10...","[170, 105, 185, 240, 130, 205, 205, 145, 95, 2..."
1,200,"[14329075, 14329075, 14329075, 14329075, 14329...","[3285270, 5253582, 9155883, 6113313, 1142869, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[9, 28, 42, 32, 0, 0, 5, 9, 100, 100, 100, 6, ...","[170, 170, 205, 155, 215, 185, 205, 195, 165, ..."
2,300,"[54090, 54100, 54110, 54120, 54125, 54135, 541...","[618910, 8793425, 8842650, 5806319, 8495320, 9...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[2, 4, 4, 5, 1, 5, 7, 1, 2, 2, 10, 6, 1, 23, 6...","[270, 130, 225, 210, 300, 210, 190, 340, 240, ..."
3,500,"[23425460, 23425835, 23426160, 23426160, 23426...","[7582363, 8350060, 2943786, 8118966, 6265477, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[20, 100, 100, 100, 100, 100, 100, 29, 10, 7, ...","[225, 210, 240, 165, 150, 245, 135, 165, 275, ..."
4,600,"[15702065, 15702065, 15702090, 15703130, 16917...","[6961851, 5856858, 7377951, 7533010, 8213481, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]","[1, 0, 0, 100, 100, 100, 100, 100, 100, 100]","[245, 215, 235, 260, 250, 280, 180, 195, 155, ..."


In [8]:
# Filtering for played_ratio_pct >= 50%
def filter_played_ratio(df):
    def filter_row(row):
        mask = [pr >= 50 for pr in row['played_ratio_pct']]
        row['timestamp'] = [ts for ts, keep in zip(row['timestamp'], mask) if keep]
        row['item_id'] = [item for item, keep in zip(row['item_id'], mask) if keep]
        row['played_ratio_pct'] = [pr for pr, keep in zip(row['played_ratio_pct'], mask) if keep]
        return row

    return df.apply(filter_row, axis=1)

filtered_organic_listen = filter_played_ratio(organic_listen)
filtered_organic_listen.head()

,uid,timestamp,item_id,is_organic,played_ratio_pct,track_length_seconds
0,100,"[287910, 1265850, 1265990, 1266160, 1266715, 1...","[5699545, 4943655, 8046835, 8661238, 4730233, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[100, 100, 100, 100, 100, 100, 100, 100, 100, ...","[170, 105, 185, 240, 130, 205, 205, 145, 95, 2..."
1,200,"[14329080, 14329080, 14329080, 16504825, 16504...","[173016, 8420846, 8085657, 2999079, 299843, 96...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[100, 100, 100, 97, 81, 100, 100, 100, 100, 10...","[170, 170, 205, 155, 215, 185, 205, 195, 165, ..."
2,300,"[2089155, 2823465, 2823810, 3001975, 3002135, ...","[6961827, 1926141, 6442688, 6442688, 5196757, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[56, 76, 100, 100, 62, 58, 100, 66, 55, 76, 10...","[270, 130, 225, 210, 300, 210, 190, 340, 240, ..."
3,500,"[23425835, 23426160, 23426160, 23426595, 23426...","[8350060, 2943786, 8118966, 6265477, 8213481, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[100, 100, 100, 100, 100, 100, 100, 100, 68, 1...","[225, 210, 240, 165, 150, 245, 135, 165, 275, ..."
4,600,"[15703130, 16917910, 16918100, 17170975, 17348...","[7533010, 8213481, 8213481, 8213481, 8213481, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]","[100, 100, 100, 100, 100, 100, 100]","[245, 215, 235, 260, 250, 280, 180, 195, 155, ..."


In [9]:
filtered_organic_listen.drop(columns=['is_organic', 'track_length_seconds'])

,uid,timestamp,item_id,played_ratio_pct
0,100,"[287910, 1265850, 1265990, 1266160, 1266715, 1...","[5699545, 4943655, 8046835, 8661238, 4730233, ...","[100, 100, 100, 100, 100, 100, 100, 100, 100, ..."
1,200,"[14329080, 14329080, 14329080, 16504825, 16504...","[173016, 8420846, 8085657, 2999079, 299843, 96...","[100, 100, 100, 97, 81, 100, 100, 100, 100, 10..."
2,300,"[2089155, 2823465, 2823810, 3001975, 3002135, ...","[6961827, 1926141, 6442688, 6442688, 5196757, ...","[56, 76, 100, 100, 62, 58, 100, 66, 55, 76, 10..."
3,500,"[23425835, 23426160, 23426160, 23426595, 23426...","[8350060, 2943786, 8118966, 6265477, 8213481, ...","[100, 100, 100, 100, 100, 100, 100, 100, 68, 1..."
4,600,"[15703130, 16917910, 16918100, 17170975, 17348...","[7533010, 8213481, 8213481, 8213481, 8213481, ...","[100, 100, 100, 100, 100, 100, 100]"
...,...,...,...,...
9233,999600,"[9760, 9760, 99155, 99490, 99705, 99910, 13893...","[246465, 7027217, 246465, 246465, 8046835, 246...","[100, 55, 100, 100, 100, 100, 100, 100, 100, 1..."
9234,999700,"[1441725, 1441945, 1442185, 1442475, 1442700, ...","[2168365, 3366945, 5180658, 4282046, 7095297, ...","[100, 100, 100, 100, 99, 99, 100, 100, 100, 10..."
9235,999800,"[15577115, 15577365, 15577510, 15578160, 15578...","[3385274, 3033749, 2651314, 7464037, 2710484, ...","[100, 100, 100, 53, 100, 100, 100, 100, 100, 1..."
9236,999900,"[5098635, 5098925, 5099225, 5099460, 5099635, ...","[9176999, 4333701, 1914019, 2241679, 3220963, ...","[64, 100, 100, 100, 100, 100, 100, 64, 100, 10..."


In [ ]:
# from datetime import timedelta
# import json
# import ast

# # === CONFIG ===
# DEBUG_MODE = False  # Set to False for full dataset
# DEBUG_USER_COUNT = 3

# # STEP 1: Work on a copy of the original
# session_df = filtered_organic_listen.copy()

# # STEP 1.5: Optional debug mode — limit to a few users
# if DEBUG_MODE:
#     sample_users = session_df['uid'].unique()[:DEBUG_USER_COUNT]
#     session_df = session_df[session_df['uid'].isin(sample_users)].copy()
#     print(f"🔍 DEBUG MODE: Using {len(sample_users)} users")

# # STEP 2: Explode session-level list columns into rows
# session_df = session_df.explode(['timestamp', 'item_id', 'played_ratio_pct'])

# # STEP 3: Convert timestamps to datetime
# session_df['timestamp'] = pd.to_numeric(session_df['timestamp'], errors='coerce')
# session_df = session_df.dropna(subset=['timestamp'])
# session_df['timestamp'] = pd.to_datetime(session_df['timestamp'], unit='s')

# # STEP 4: Sort
# session_df = session_df.sort_values(by=['uid', 'timestamp'])

# # STEP 5: Build mappings BEFORE splitting
# all_uids = session_df['uid'].dropna().unique()
# all_items = session_df['item_id'].dropna().unique()

# uid2idx = {str(uid): i for i, uid in enumerate(sorted(all_uids))}
# item2idx = {str(item): i + 1 for i, item in enumerate(sorted(all_items))}  # reserve 0 for padding

# # STEP 6: Apply mappings
# session_df['uid'] = session_df['uid'].map(lambda x: uid2idx.get(str(x), None))
# session_df['item_id'] = session_df['item_id'].map(lambda x: item2idx.get(str(x), None))
# session_df = session_df.dropna(subset=['uid', 'item_id'])


In [ ]:

# # Convert to int
# session_df['uid'] = session_df['uid'].astype(int)
# session_df['item_id'] = session_df['item_id'].astype(int)

# # STEP 7: Time-based split
# min_time = session_df['timestamp'].min()
# train_end = min_time + timedelta(days=299)
# val_start = train_end
# val_end = val_start + timedelta(days=1)
# test_start = val_end + timedelta(minutes=30)
# test_end = test_start + timedelta(days=1)

# train_data = session_df[session_df['timestamp'] < train_end]
# val_data = session_df[(session_df['timestamp'] >= val_start) & (session_df['timestamp'] < val_end)]
# test_data = session_df[(session_df['timestamp'] >= test_start) & (session_df['timestamp'] < test_end)]

# # STEP 8: Generate training set (sliding window)
# train_rows = []

# for uid, group in train_data.groupby('uid'):
#     items = group['item_id'].tolist()
#     for i in range(1, len(items)):
#         history = items[max(0, i - 20):i]
#         next_item = items[i]
#         train_rows.append({'user_id': uid, 'user_history': history, 'next_item': next_item})

# train_df = pd.DataFrame(train_rows)

# # STEP 9: Generate validation set
# val_rows = []

# for uid, group in val_data.groupby('uid'):
#     train_items = train_data[train_data['uid'] == uid]['item_id'].tolist()
#     if not train_items:
#         continue
#     history = train_items[-20:]
#     predicting_items = group['item_id'].tolist()
#     if predicting_items:
#         val_rows.append({'user_id': uid, 'given_user_history': history, 'predicting_items': predicting_items})

# val_df = pd.DataFrame(val_rows)

# # STEP 10: Generate test set (only users with history)
# test_rows = []

# for uid, group in test_data.groupby('uid'):
#     prior_items = session_df[(session_df['uid'] == uid) &(session_df['timestamp'] < val_end)]['item_id'].tolist()
#     if not prior_items:
#         continue
#     history = prior_items[-20:]
#     predicting_items = group['item_id'].tolist()
#     if predicting_items:
#         test_rows.append({'user_id': uid, 'given_user_history': history, 'predicting_items': predicting_items})

# test_df = pd.DataFrame(test_rows)

# # STEP 11: Save mappings (convert keys to str for JSON)
# uid2idx_str = {str(k): int(v) for k, v in uid2idx.items()}
# item2idx_str = {str(k): int(v) for k, v in item2idx.items()}

# with open("uid2idx.json", "w") as f:
#     json.dump(uid2idx_str, f)

# with open("item2idx.json", "w") as f:
#     json.dump(item2idx_str, f)

# # STEP 12: Save datasets as TSV
# train_df.to_csv("training.tsv", sep="\t", index=False)
# val_df.to_csv("validation.tsv", sep="\t", index=False)
# test_df.to_csv("test.tsv", sep="\t", index=False)

# # STEP 13: Report
# print("✅ Done!")
# print("Train size:", train_df.shape)
# print("Validation size:", val_df.shape)
# print("Test size:", test_df.shape)
# print("Unique users:", len(uid2idx))
# print("Unique items:", len(item2idx))


✅ Done!
Train size: (13975750, 3)
Validation size: (3222, 3)
Test size: (3190, 3)
Unique users: 8798
Unique items: 475910


In [14]:
lengths = test_df['predicting_items'].apply(len)
print("🔍 Test prediction stats:")
print("- Number of users:", len(lengths))
print("- Avg items to predict:", lengths.mean())
print("- Min items:", lengths.min())
print("- Max items:", lengths.max())


🔍 Test prediction stats:
- Number of users: 3190
- Avg items to predict: 15.502194357366772
- Min items: 1
- Max items: 185


In [15]:
# Add a helper column for length
test_df['predicting_items_len'] = test_df['predicting_items'].apply(len)

# Find the row with the max length
max_row = test_df.loc[test_df['predicting_items_len'].idxmax()]

# Print the full row
print("🔍 Test user with the most items to predict:")
print(max_row)


🔍 Test user with the most items to predict:
uid                                                                  5551
history                 [460722, 122085, 409111, 447195, 176655, 30224...
predicting_items        [182180, 286557, 411924, 431706, 425662, 27881...
predicting_items_len                                                  185
Name: 2020, dtype: object


In [16]:
from collections import Counter

item_counts = Counter(max_row['predicting_items'])
repeats = {k: v for k, v in item_counts.items() if v > 1}
print(f"🔁 Repeated items (count > 1): {repeats}")


🔁 Repeated items (count > 1): {}


In [12]:
import pandas as pd
from datetime import timedelta
import json

# === CONFIG ===
DEBUG_MODE = False
DEBUG_USER_COUNT = 3

# === UTILITIES ===
def dedup(seq):
    return list(dict.fromkeys(seq)) if isinstance(seq, list) else seq

# STEP 1: Copy original data
session_df = filtered_organic_listen.copy()

# STEP 1.5: Debug mode
if DEBUG_MODE:
    sample_users = session_df['uid'].unique()[:DEBUG_USER_COUNT]
    session_df = session_df[session_df['uid'].isin(sample_users)].copy()
    print(f"🔍 DEBUG MODE: Using {len(sample_users)} users")

# STEP 2: Explode list-columns
session_df = session_df.explode(['timestamp', 'item_id', 'played_ratio_pct'])

# STEP 3: Convert timestamps
session_df['timestamp'] = pd.to_numeric(session_df['timestamp'], errors='coerce')
session_df = session_df.dropna(subset=['timestamp'])
session_df['timestamp'] = pd.to_datetime(session_df['timestamp'], unit='s')

# STEP 4: Sort
session_df = session_df.sort_values(by=['uid', 'timestamp'])

# STEP 5: Remap IDs
all_uids = session_df['uid'].dropna().unique()
all_items = session_df['item_id'].dropna().unique()

uid2idx = {str(uid): i for i, uid in enumerate(sorted(all_uids))}
item2idx = {str(item): i + 1 for i, item in enumerate(sorted(all_items))}

session_df['uid'] = session_df['uid'].map(lambda x: uid2idx.get(str(x)))
session_df['item_id'] = session_df['item_id'].map(lambda x: item2idx.get(str(x)))
session_df = session_df.dropna(subset=['uid', 'item_id'])


In [ ]:

session_df['uid'] = session_df['uid'].astype(int)
session_df['item_id'] = session_df['item_id'].astype(int)

# STEP 6: Time ranges
min_time = session_df['timestamp'].min()
train_end = min_time + timedelta(days=299)
val_start = train_end
val_end = val_start + timedelta(days=1)
test_start = val_end + timedelta(minutes=30)
test_end = test_start + timedelta(days=1)

# STEP 7: Split data
train_data = session_df[session_df['timestamp'] < train_end]
val_data = session_df[(session_df['timestamp'] >= val_start) & (session_df['timestamp'] < val_end)]
test_data = session_df[(session_df['timestamp'] >= test_start) & (session_df['timestamp'] < test_end)]

# STEP 8: Train set (dedup first, then slice) --> Normal case
# train_rows = []

# for uid, group in train_data.groupby('uid'):
#     items = group['item_id'].tolist()
#     for i in range(1, len(items)):
#         history = dedup(items[:i])[-20:]  # ✅ correct order
#         next_item = items[i]
#         train_rows.append({'uid': uid, 'history': history, 'next_item': next_item})

# train_df = pd.DataFrame(train_rows)
# STEP 8: Train set (fixed windows of 20) ----> BERC4REC
train_rows = []

for uid, group in train_data.groupby('uid'):
    items = dedup(group['item_id'].tolist())
    
    # Split items into chunks of 20
    for i in range(0, len(items) - 40 + 1, 20):  # Ensure we have full 20 for history and next
        history_chunk = items[i:i+20]
        next_chunk = items[i+20:i+40]
        if len(next_chunk) == 20:  # Ensure both chunks are full
            train_rows.append({
            'uid': uid,
            'history': history_chunk,  # growing history
            })

train_df = pd.DataFrame(train_rows)


# STEP 9: Validation set
val_rows = []

for uid, group in val_data.groupby('uid'):
    train_items = train_data[train_data['uid'] == uid]['item_id'].tolist()
    if not train_items:
        continue
    history = dedup(train_items)[-20:]
    predicting_items = dedup(group['item_id'].tolist())
    if predicting_items:
        val_rows.append({'uid': uid, 'history': history, 'predicting_items': predicting_items})

val_df = pd.DataFrame(val_rows)

# STEP 10: Test set
test_rows = []

for uid, group in test_data.groupby('uid'):
    prior_items = session_df[
        (session_df['uid'] == uid) &
        (session_df['timestamp'] < val_end)  # ✅ ends before gap
    ]['item_id'].tolist()
    if not prior_items:
        continue
    history = dedup(prior_items)[-20:]
    predicting_items = dedup(group['item_id'].tolist())
    if predicting_items:
        test_rows.append({'uid': uid, 'history': history, 'predicting_items': predicting_items})

test_df = pd.DataFrame(test_rows)

# STEP 11: Save mappings
uid2idx_str = {str(k): int(v) for k, v in uid2idx.items()}
item2idx_str = {str(k): int(v) for k, v in item2idx.items()}

with open("../../data_bert4rec/uid2idx.json", "w") as f:
    json.dump(uid2idx_str, f)

with open("../../data_bert4rec/item2idx.json", "w") as f:
    json.dump(item2idx_str, f)

# STEP 12: Save datasets
train_df.to_csv("../../data_bert4rec/training.tsv", sep="\t", index=False)
val_df.to_csv("../../data_bert4rec/validation.tsv", sep="\t", index=False)
test_df.to_csv("../../data_bert4rec/test.tsv", sep="\t", index=False)

# STEP 13: Summary
print("✅ Done!")
print("Train size:", train_df.shape)
print("Validation size:", val_df.shape)
print("Test size:", test_df.shape)
print("Unique users:", len(uid2idx))
print("Unique items:", len(item2idx))

# STEP 14: Prediction stats
test_lengths = test_df['predicting_items'].apply(len)
print("🔍 Test predicting items:")
print("- Avg length:", test_lengths.mean())
print("- Max length:", test_lengths.max())
print("- Min length:", test_lengths.min())


✅ Done!
Train size: (164099, 2)
Validation size: (3222, 3)
Test size: (3190, 3)
Unique users: 8798
Unique items: 475910
🔍 Test predicting items:
- Avg length: 15.502194357366772
- Max length: 185
- Min length: 1


# Bert Preprocessing

In [1]:
import pandas as pd
import ast


In [2]:
# load Yambda training.tsv and print first 5 rows
yambda_train_df = pd.read_csv("training.tsv", sep="\t")
print("Yambda Training Data Sample:")
print(yambda_train_df.head())

Yambda Training Data Sample:
   uid                                   history  next_item
0    0                                  [289700]     251076
1    0                          [289700, 251076]     408129
2    0                  [289700, 251076, 408129]     439236
3    0          [289700, 251076, 408129, 439236]     240032
4    0  [289700, 251076, 408129, 439236, 240032]     119231


In [3]:
# Convert stringified list in 'history' to actual list
yambda_train_df["history"] = yambda_train_df["history"].apply(ast.literal_eval)

# Add history length and sort so last interaction is at bottom per user
yambda_train_df["history_length"] = yambda_train_df["history"].apply(len)
yambda_train_df = yambda_train_df.sort_values(by=["uid", "history_length"])

# Get the last row for each user
last_entries = yambda_train_df.groupby("uid").tail(1).copy()

# Concatenate history and next_item
last_entries["user_history"] = last_entries.apply(
    lambda row: row["history"] + [row["next_item"]], axis=1
)

# Convert the list to a string like '[1, 2, 3, 4]'
last_entries["user_history"] = last_entries["user_history"].apply(str)


# 1. Keep both the user ID and the history columns
# 2. Reset the index to be clean (0, 1, 2...)
final_df = last_entries[["uid", "user_history"]].reset_index(drop=True)

# 3. Rename the 'uid' column to 'user_id' for the final output
final_df.rename(columns={"uid": "user_id"}, inplace=True)



final_df.to_csv("bert4rec_training.csv", index=False)

print("Saved BERT4Rec training file with shape:", final_df.shape)

Saved BERT4Rec training file with shape: (8717, 2)


In [6]:
# load Yambda bert4rec_training.csv and print first 5 rows
bert4rec_train_df = pd.read_csv("bert4rec_training.csv")
print("BERT4Rec Training Data Sample:")
print(bert4rec_train_df.head())
# print the datatypes of the columns
print("Data Types:")
print(bert4rec_train_df.dtypes)
# make the csv a tsv 
bert4rec_train_df.to_csv("bert4rec_training.tsv", sep="\t", index=False)
print("Saved BERT4Rec training file as TSV with shape:", bert4rec_train_df.shape)

BERT4Rec Training Data Sample:
   user_id                                       user_history
0        0  [123290, 405939, 219647, 137930, 50091, 416042...
1        1  [305246, 134618, 248722, 70629, 402235, 190030...
2        2  [304615, 402992, 363395, 80024, 50091, 448377,...
3        3  [423485, 149657, 411875, 318178, 416656, 11754...
4        4                           [382183, 416656, 416656]
Data Types:
user_id          int64
user_history    object
dtype: object
Saved BERT4Rec training file as TSV with shape: (8717, 2)


In [8]:
# load Yambda bert4rec tsv and print first 5 rows
bert4rec_train_tsv_df = pd.read_csv("bert4rec_training.tsv", sep="\t")
print("BERT4Rec Training TSV Data Sample:")
print(bert4rec_train_tsv_df.head())

BERT4Rec Training TSV Data Sample:
   user_id                                       user_history
0        0  [123290, 405939, 219647, 137930, 50091, 416042...
1        1  [305246, 134618, 248722, 70629, 402235, 190030...
2        2  [304615, 402992, 363395, 80024, 50091, 448377,...
3        3  [423485, 149657, 411875, 318178, 416656, 11754...
4        4                           [382183, 416656, 416656]


In [4]:
dataframes['artist_item_mapping.parquet'].head(30)

,artist_id,item_id
0,1,953587
1,1,1921481
2,1,2659068
3,1,2740764
4,1,6767564
5,1,7382780
6,2,1363435
7,2,1995921
8,2,3117333
9,2,4631303


In [ ]:
# load item2idx.json and print first 5 rows
import json
with open("./item2idx.json", "r") as f:
    item2idx = json.load(f)

Item2Idx Mapping Sample:
[('26', 1), ('43', 2), ('50', 3), ('71', 4), ('105', 5)]


In [12]:
import pandas as pd
import json

# Assume item2idx is already defined, e.g.:
# item2idx = {'1001': 0, '1002': 1, ...}

# Assume dataframes is a dictionary with a key 'artist_item_mapping.parquet'
# which contains a pandas DataFrame loaded from a .parquet file
# For example:
# dataframes = {
#     'artist_item_mapping.parquet': pd.read_parquet('artist_item_mapping.parquet')
# }

# Step 1: Extract the mapping from item_id to artist_id from the DataFrame
artist_df = dataframes['artist_item_mapping.parquet'][['item_id', 'artist_id']]

# Ensure item_id is string to match with item2idx keys (if they are strings)
artist_mapping = dict(zip(artist_df['item_id'].astype(str), artist_df['artist_id']))

# Step 2: Build item_to_artist mapping
item_to_artist = {}
for item in item2idx:
    artist = artist_mapping.get(str(item))  # Ensure type match
    if artist is not None:
        item_to_artist[item] = [str(artist)]

# Step 3: Print first 5 entries
print("First 5 item-to-artist mappings:")
for item, artist in list(item_to_artist.items())[:5]:
    print(f"Item ID: {item}, Artist: {artist}")

# Step 4: Save the item_to_artist mapping to a JSON file
with open("id2cat.json", "w") as f:
    json.dump(item_to_artist, f)


First 5 item-to-artist mappings:
Item ID: 26, Artist: ['211987']
Item ID: 43, Artist: ['1169269']
Item ID: 50, Artist: ['641376']
Item ID: 71, Artist: ['316170']
Item ID: 105, Artist: ['570610']
